# 🧪 Đánh giá mô hình LLM Multi-Agent (Sequential Adversarial Pipeline) với ViFactCheck

Notebook này được thiết kế để chạy trên **Google Colab**. Mục đích là load tập test của ViFactCheck và sử dụng Pipeline Đa Đặc Vụ (sử dụng model của NVIDIA hoặc một mô hình LLM bất kỳ) để phân tích, chạy qua 8 bước, và phân loại đoạn text có phải là tin giả hay không. Cuối cùng sẽ thống kê các metrics đánh giá (Accuracy, F1-Score...).

## 📥 Bước 1: Clone Repository từ GitHub và chuyển thư mục làm việc

In [ ]:
!rm -rf fake-news-detection
!git clone https://github.com/phidinhmanh/fake-news-detection.git
%cd fake-news-detection

## 🛠️ Bước 2: Cài đặt Dependencies

In [ ]:
!pip install -q underthesea transformers datasets lancedb sentence-transformers google-generativeai langdetect wordcloud plotly streamlit pyngrok python-dotenv huggingface_hub scikit-learn langchain-nvidia-ai-endpoints
!pip install -q pydantic

## ⚙️ Bước 3: Thiết lập API Keys

Bắt buộc cần `NVIDIA_API_KEY` để mô hình Multi-Agent có thể gọi LLMs thông qua dịch vụ của NVIDIA.

In [ ]:
import os
import sys
import getpass
from pathlib import Path
from huggingface_hub import login

ROOT_DIR = Path("..").resolve() if Path(".").resolve().name == "notebooks" else Path(".").resolve()
sys.path.append(str(ROOT_DIR))
print(f"Project Root: {ROOT_DIR}")

def setup_nvidia_api():
    key = os.getenv("NVIDIA_API_KEY")
    if not key:
        try:
            from google.colab import userdata
            key = userdata.get('NVIDIA_API_KEY')
        except:
            pass
    if not key:
        print("🔑 Vui lòng nhập NVIDIA_API_KEY (NVIDIA NIM API):")
        key = getpass.getpass()
    if key:
        os.environ["NVIDIA_API_KEY"] = key
        print("✅ NVIDIA_API_KEY đã được thiết lập.")
    else:
        print("⚠️ Không có NVIDIA_API_KEY - Hệ thống sẽ chạy ở chế độ MOCK (Dữ liệu cố định).")

def setup_hf_token():
    token = os.getenv("HF_TOKEN")
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get('HF_TOKEN')
        except:
            pass
    if not token:
        print("🔑 Vui lòng nhập HF_TOKEN (Để load tập dữ liệu private nếu cần):")
        token = getpass.getpass()
    if token:
        os.environ["HF_TOKEN"] = token
        login(token=token)
        print("✅ Đã đăng nhập Hugging Face Hub.")
    else:
        print("⚠️ Bỏ qua cấu hình Hugging Face Token.")

setup_nvidia_api()
setup_hf_token()

## 📂 Bước 4: Tải dữ liệu ViFactCheck

Tiến hành download tập dữ liệu, làm sạch và gộp vào thư mục `dataset/processed/`.

In [ ]:
!python dataset/download_datasets.py
!python dataset/preprocess_vietnamese.py
!python dataset/merge_datasets.py

## 🧠 Bước 5: Đánh giá quá trình (Evaluation)

Do gọi API NVIDIA / LLMs tốn nhiều chi phí token và bị giới hạn **Rate Limit** trên free tier, chúng ta sẽ chọn một mẫu (`NUM_SAMPLES = 20`) để demo.
Bạn có thể tăng giá trị này lên nếu cần test với quy mô lớn hơn.
Bạn cũng có thể thay đổi model hoặc provider từ `config.py` trước khi chạy.

In [ ]:
import pandas as pd
import numpy as np
import os
from config import DATASET_PROCESSED_DIR
from tqdm.auto import tqdm
import time
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sequential_adversarial.pipeline import SequentialAdversarialPipeline

test_path = DATASET_PROCESSED_DIR / 'test.csv'
if not test_path.exists():
    print("❌ Lỗi: Không tìm thấy test.csv. Vui lòng kiểm tra lại bước 4.")
else:
    df_test = pd.read_csv(test_path)
    print(f"✅ Đã load tập test: {len(df_test)} mẫu.")

    # Lọc bỏ các dòng lỗi thiếu dữ liệu
    df_test = df_test.dropna(subset=['text', 'label'])
    
    # LLM inference chạy khá chậm nên có thể lấy n mẫu để thử nghiệm
    NUM_SAMPLES = 20 
    # Bạn có thể lọc tỷ lệ 50-50 class cân bằng để test khách quan hơn
    df_fake = df_test[df_test['label'] == 'fake'].sample(n=min(NUM_SAMPLES // 2, sum(df_test['label'] == 'fake')), random_state=42)
    df_real = df_test[df_test['label'] == 'real'].sample(n=min(NUM_SAMPLES - len(df_fake), sum(df_test['label'] == 'real')), random_state=42)
    df_subset = pd.concat([df_fake, df_real]).sample(frac=1, random_state=42).reset_index(drop=True)

    print(f"Tập dữ liệu đánh giá: {len(df_subset)} samples (Fake: {len(df_fake)}, Real: {len(df_real)})")
    
    # Đảm bảo cấu hình là dùng NVIDIA API
    os.environ["LLM_PROVIDER"] = "nvidia"
    
    # Khởi tạo Pipeline (Bật mock nếu ko có API)
    has_key = os.getenv("NVIDIA_API_KEY") is not None
    pipeline = SequentialAdversarialPipeline(mock=not has_key)

    y_true = []
    y_pred = []
    results = []
    
    for idx, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc="Evaluating Multi-Agent Pipeline"):
        text = str(row['text'])
        true_label = str(row['label']).strip().lower() # 'real' hoặc 'fake'
        
        # Chạy pipeline 8 stage của bạn
        try:
            result = pipeline.run(text)
            if result.verity_report:
                conclusion = result.verity_report.conclusion.lower()
                # Map conclusion ('true', 'false', 'mixed') -> 'real', 'fake'
                # Theo logic của pipeline, Tin giả sẽ là False or Mixed.
                if conclusion == 'true':
                    pred_label = 'real'
                elif conclusion in ['false', 'mixed']:
                    pred_label = 'fake'
                else:
                    pred_label = 'fake'
            else:
                pred_label = 'fake' # An toàn nếu thất bại
                
        except Exception as e:
            print(f"⚠️ Lỗi xử lý ở mẫu {idx}: {e}")
            pred_label = 'fake' # Fallback
            result = None
            
        y_true.append(true_label)
        y_pred.append(pred_label)
        
        results.append({
            'text_snippet': text[:150] + '...',
            'true_label': true_label,
            'pred_label': pred_label,
            'confidence': result.verity_report.confidence if (result and result.verity_report) else 0,
            'reasoning_summary': result.verity_report.evidence_summary if (result and result.verity_report) else "N/A"
        })
        
        # Rate limited delay để trách lỗi quá tải API
        if has_key:
            time.sleep(3)
            
    print("\n" + "="*50)
    print("📊 BÁO CÁO ĐÁNH GIÁ (CLASSIFICATION REPORT)")
    print("="*50)
    
    # In các chỉ số hiệu năng cơ bản
    labels = ['fake', 'real']
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))
    
    # Vẽ Confusion Matrix
    print("\n🧮 CONFUSION MATRIX")
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    df_cm = pd.DataFrame(cm, index=[f"True {l}" for l in labels], columns=[f"Pred {l}" for l in labels])
    print(df_cm)
    
    # Hiển thị vài ví dụ phân loại sai (Nếu Có)
    print("\n❌ MỘT SỐ VÍ DỤ PHÂN LOẠI SAI CẦN LƯU Ý:")
    df_res = pd.DataFrame(results)
    wrong_cases = df_res[df_res['true_label'] != df_res['pred_label']]
    if not wrong_cases.empty:
        for _, r in wrong_cases.head(3).iterrows():
            print(f"- True: {r['true_label'].upper()} | Pred: {r['pred_label'].upper()} | Conf: {r['confidence']:.2f}")
            print(f"  [Text]: {r['text_snippet']}")
            print(f"  [LLM Lý do]: {r['reasoning_summary']}\n")
    else:
        print("🔥 Tuyệt vời! Mô hình phân loại chuẩn xác 100% trên tập này!")
